## Preparation:Generate the Primary Transactions File 

This section reads only 1000 Sales Records.csv. It selects the first 500 data rows (excluding the header), in their original order, and writes data/raw/transactions_500.csv.
All 14 original column names and all selected values are preserved, including whitespace



In [4]:
"""Select the first 500 data rows using only the original CSV."""
import csv
from itertools import islice
from pathlib import Path

ROOT = Path.cwd()


def prepare():
    # Read strings without trimming, renaming, type conversion or enrichment.
    source = ROOT / "data" / "raw" / "1000 Sales Records.csv"
    destination = ROOT / "data" / "raw" / "transactions_500.csv"
    #destination.parent.mkdir(parents=True, exist_ok=True)

    with source.open(newline="", encoding="utf-8-sig") as handle:
        reader = csv.reader(handle)
        header = next(reader)
        rows = list(islice(reader, 500))

    assert len(rows) == 500, "The source must contain at least 500 data rows."

    with destination.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.writer(handle)
        writer.writerow(header)
        writer.writerows(rows)

    print(
        f"Saved {len(rows)} rows and {len(header)} original columns "
        f"to {destination.name}."
    )


prepare()

Saved 500 rows and 14 original columns to transactions_500.csv.


## Hello, Data!

Load the newly created transactions_500 CSV and display its first three rows. Read as strings to preserve raw formatting. Confirm that its headers and values exactly match the first 500 rows of the original source.

In [5]:
import json
import random
from collections import namedtuple
from decimal import Decimal, ROUND_HALF_UP
import pandas as pd
from IPython.display import display, Markdown

ROOT / "data" / "raw" / "1000 Sales Records.csv"

RAW = ROOT / "data" / "raw" 
META = ROOT / "data" / "metadata" 
OUT = ROOT / "data" / "processed"
OUT.mkdir(exist_ok=True)
original = pd.read_csv(RAW / "1000 Sales Records.csv", dtype=str, keep_default_na=False)
primary = pd.read_csv(RAW / "transactions_500.csv", dtype=str, keep_default_na=False)
pd.testing.assert_frame_equal(primary, original.head(500).reset_index(drop=True))
assert primary.shape == (500, 14)
display(primary.head(3))

,Region,Country,Item Type,Sales Channel,Order Priority,Order Date,Order ID,Ship Date,Units Sold,Unit Price,Unit Cost,Total Revenue,Total Cost,Total Profit
0,Middle East and North Africa,Libya,Cosmetics,Offline,M,10/18/2014,686800706,10/31/2014,8446,437.20,263.33,3692591.20,2224085.18,1468506.02
1,North America,Canada,Vegetables,Online,M,11/7/2011,185941302,12/8/2011,3018,154.06,90.93,464953.08,274426.74,190526.34
2,Middle East and North Africa,Libya,Baby Food,Offline,C,10/31/2016,246222341,12/9/2016,1517,255.28,159.42,387259.76,241840.14,145419.62


##  Pick the Right Container

Choosing a dictionary for each transaction because its named fields can be updated during cleaning, whereas a namedtuple suits a fixed, immutable snapshot. A list preserves all transactions in order, while a set removes repeated country names for a distinct-country count.

In [6]:

transaction = primary.iloc[0].to_dict()
Snapshot = namedtuple('Snapshot', ['order_id', 'product', 'price'])
snapshot = Snapshot(transaction['Order ID'], transaction['Item Type'], transaction['Unit Price'])
transactions = primary.to_dict(orient='records')
unique_countries = set(primary['Country'].str.strip())
print('Dictionary product:', transaction['Item Type'])
print('Namedtuple product:', snapshot.product)
print('Records in list:', len(transactions))
print('Distinct countries:', len(unique_countries))

Dictionary product: Cosmetics
Namedtuple product: Cosmetics
Records in list: 500
Distinct countries: 171


## Implement Functions and  Data structure

make_record() populates a dictionary. SalesBatch stores a list of records and provides reusable clean() and total() methods, keeping the source DataFrame unchanged.

In [7]:
def make_record(row):
    """Turn a pandas row into a plain dictionary."""
    return {field: value for field, value in row.items()}


def money(value):
    """Round monetary amounts to cents using decimal half-up rounding."""
    return float(Decimal(str(value)).quantize(Decimal('0.01'), rounding=ROUND_HALF_UP))


class SalesBatch:
    """A list of transaction dictionaries plus their cleaning and total behavior."""
    def __init__(self, records):
        self.records = [dict(record) for record in records]

    def total(self):
        """Return gross revenue from unit prices and quantities."""
        amount = sum((Decimal(str(r['Unit Price'])) * Decimal(str(r['Units Sold']))
                      for r in self.records), Decimal('0'))
        return money(amount)

    def clean(self):
        """Repair the documented cases and validate the original sales fields."""
        frame = pd.DataFrame(self.records).drop_duplicates().copy()
        for field in frame.columns:
            frame[field] = frame[field].map(lambda x: x.strip() if isinstance(x, str) else x)
        quantity = pd.to_numeric(frame['Units Sold'], errors='raise')
        if not ((quantity > 0) & (quantity % 1 == 0)).all():
            raise ValueError('Quantity must be a positive whole number.')
        frame['Units Sold'] = quantity.astype('int64')
        missing_price = frame['Unit Price'].isna() | frame['Unit Price'].eq('')
        for index in frame.index[missing_price]:
            gross = Decimal(str(frame.at[index, 'Total Revenue']))
            units = Decimal(str(frame.at[index, 'Units Sold']))
            price = (gross / units).quantize(Decimal('0.01'), rounding=ROUND_HALF_UP)
            if price * units != gross:
                raise ValueError('Cannot recover a price that reconciles to source revenue.')
            frame.at[index, 'Unit Price'] = str(price)
        for field in ['Unit Price', 'Unit Cost', 'Total Revenue', 'Total Cost', 'Total Profit']:
            frame[field] = pd.to_numeric(frame[field], errors='raise')
        for field in ['Order Date', 'Ship Date']:
            if not pd.api.types.is_datetime64_any_dtype(frame[field]):
                frame[field] = pd.to_datetime(frame[field], format='%m/%d/%Y', errors='raise')
        if frame.isna().any().any() or frame.eq('').any().any():
            raise ValueError('Unresolved missing data.')
        if frame['Order ID'].duplicated().any():
            raise ValueError('Conflicting records share an Order ID.')
        if not frame['Unit Price'].gt(0).all():
            raise ValueError('Unit prices must be positive.')
        if (frame['Ship Date'] < frame['Order Date']).any():
            raise ValueError('Shipment precedes order date.')
        expected = frame['Unit Price'] * frame['Units Sold']
        if not (expected - frame['Total Revenue']).abs().lt(0.005).all():
            raise ValueError('Gross revenue does not reconcile.')
        frame = frame.reset_index(drop=True)
        self.records = frame.to_dict(orient='records')
        return frame

first_record = make_record(primary.iloc[0])
demo = SalesBatch([first_record])

print('Single-order gross revenue:', demo.total())

Single-order gross revenue: 3692591.2


## Bulk Loaded

Map the complete DataFrame into a list of dictionaries and load the class.Using a list preserves every order; a dictionary keyed by a nonunique field could overwrite transactions.

In [8]:
records = primary.to_dict(orient='records')
batch = SalesBatch(records)
assert len(batch.records) == 500
print('Loaded', len(batch.records), 'transaction dictionaries.')
display(pd.DataFrame(batch.records[:3]))

Loaded 500 transaction dictionaries.


,Region,Country,Item Type,Sales Channel,Order Priority,Order Date,Order ID,Ship Date,Units Sold,Unit Price,Unit Cost,Total Revenue,Total Cost,Total Profit
0,Middle East and North Africa,Libya,Cosmetics,Offline,M,10/18/2014,686800706,10/31/2014,8446,437.20,263.33,3692591.20,2224085.18,1468506.02
1,North America,Canada,Vegetables,Online,M,11/7/2011,185941302,12/8/2011,3018,154.06,90.93,464953.08,274426.74,190526.34
2,Middle East and North Africa,Libya,Baby Food,Offline,C,10/31/2016,246222341,12/9/2016,1517,255.28,159.42,387259.76,241840.14,145419.62


## Quick Profiling
Calculate minimum, mean and maximum price.This is an initial check of what data contains before cleaning it .Here we examine prices ,geographic coverage, missing values, and duplicates using the 500-row subset.

In [10]:
prices = pd.to_numeric(primary['Unit Price'], errors='raise')
display(prices.agg(['min', 'mean', 'max']).rename('Unit Price'))
print('Unique countries:', len(set(primary['Country'].str.strip())))
print('Missing source cells:', int(primary.eq('').sum().sum()))
print('Exact source duplicates:', int(primary.duplicated().sum()))

min       9.33000
mean    274.29506
max     668.27000
Name: Unit Price, dtype: float64

Unique countries: 171
Missing source cells: 0
Exact source duplicates: 0


## Spot the Grime

The source has 22 country values with surrounding whitespace, but no missing values or exact duplicate rows in the selected batch. To demonstrate three distinct cleaning cases , lets try to make the data dirty by adding one blank unit price and one exact duplicate row to working copy only; these two cases are simulated and are not source defects.
The duplicate is an extra copy of the second row; the blank price is in the first row. The primary CSV stays at 500 unchanged rows. The temporary working batch has 501 rows, returning to 500 after cleaning. Counts describe separate cases, not an additive total of affected orders.

In [9]:
def issue_counts(frame):
    """Use the same measures before and after cleaning."""
    country = frame['Country'].fillna('')
    return pd.Series({
        'country whitespace (source)': int(country.ne(country.str.strip()).sum()),
        'missing unit price (simulated)': int((frame['Unit Price'].isna() | frame['Unit Price'].eq('')).sum()),
        'exact duplicate rows (simulated)': int(frame.duplicated().sum()),
    })

working = pd.DataFrame(batch.records)
cleaning_log = [
    {'case': 'blank unit price', 'order_id': working.loc[0, 'Order ID'],
     'original': working.loc[0, 'Unit Price'], 'teaching_value': ''},
    {'case': 'duplicate row', 'order_id': working.loc[1, 'Order ID'],
     'action': 'append an exact copy of the second row'},
]
working.loc[0, 'Unit Price'] = ''
working = pd.concat([working, working.iloc[[1]]], ignore_index=True)
batch = SalesBatch(working.to_dict(orient='records'))
before = issue_counts(working)
assert (before > 0).all()
display(before.to_frame('before'))
display(working.iloc[[0, 1, -1]][['Order ID', 'Country', 'Unit Price']])
(OUT / 'cleaning_demo_log.json').write_text(json.dumps(cleaning_log, indent=2) + '\n')

,before
country whitespace (source),22
missing unit price (simulated),1
exact duplicate rows (simulated),1


,Order ID,Country,Unit Price
0,686800706,Libya,
1,185941302,Canada,154.06
500,185941302,Canada,154.06


246

## Cleaning Rules

Call clean() to remove the exact duplicate, trim whitespace, recover the blank price from the source arithmetic, and convert numeric/date types. Show before/after counts, confirm all 500 original order IDs remain, and verify that cleaning a second time changes nothing.

In [10]:
cleaned_source = batch.clean()
after = issue_counts(cleaned_source)
display(pd.DataFrame({'before': before, 'after': after}))
assert after.sum() == 0 and len(cleaned_source) == 500
assert cleaned_source['Order ID'].tolist() == primary['Order ID'].tolist()
assert cleaned_source.loc[0, 'Unit Price'] == float(primary.loc[0, 'Unit Price'])
pd.testing.assert_frame_equal(cleaned_source, batch.clean())
print('Gross revenue after cleaning:', batch.total())
display(cleaned_source.dtypes.to_frame('type'))

,before,after
country whitespace (source),22,0
missing unit price (simulated),1,0
exact duplicate rows (simulated),1,0


Gross revenue after cleaning: 711192227.21


,type
Region,str
Country,str
Item Type,str
Sales Channel,str
Order Priority,str
Order Date,datetime64[us]
Order ID,str
Ship Date,datetime64[us]
Units Sold,int64
Unit Price,float64


##  Transformations

Rename columns to consistent names by writing names in lowercase and separating words with underscores _ and encode Sales Channel as is_online (Online = 1, Offline = 0) to demonstrate a categorical to numerical transformation. This transformation uses only original fields

In [11]:
rename_map = {
    'Region': 'region', 'Country': 'country', 'Item Type': 'product',
    'Sales Channel': 'sales_channel', 'Order Priority': 'order_priority',
    'Order Date': 'date', 'Order ID': 'order_id', 'Ship Date': 'ship_date',
    'Units Sold': 'quantity', 'Unit Price': 'price', 'Unit Cost': 'unit_cost',
    'Total Revenue': 'gross_revenue', 'Total Cost': 'total_cost', 'Total Profit': 'gross_profit',
}
cleaned = cleaned_source.rename(columns=rename_map).copy()
channel_codes = cleaned['sales_channel'].map({'Online': 1, 'Offline': 0})
assert channel_codes.notna().all(), 'Unexpected sales channel.'
cleaned['is_online'] = channel_codes.astype('int64')
display(cleaned[['order_id', 'sales_channel', 'is_online', 'price', 'quantity']].head())

,order_id,sales_channel,is_online,price,quantity
0,686800706,Offline,0,437.20,8446
1,185941302,Online,1,154.06,3018
2,246222341,Offline,0,255.28,1517
3,161442649,Offline,0,205.70,3322
4,645713555,Offline,0,9.33,9845


## Feature Engineering

 Access the secondary metadata here that we have downloaded "geonames_countryInfo.txt" and "country_aliases.json". Read GeoNames, reconcile country-name aliases, and validate a many-to-one join so rows cannot multiply. Assign each country's capital as a synthetic shipping city, then generate fictional customer IDs and coupons with seed 42. Customer IDs may repeat but remain within one country.We stick to seed 42 that gives the same synthetic data, making results easier to test and reproduce.Without a fixed seed, generated values could change between runs and so could your revenue results.In this case we have chosen 42

Map coupons NONE/SAVE20/SAVE30 to 0/20/30% discounts and calculate net revenue, rounding each line to cents. days_since_purchase uses one day after the latest order as a fixed reference, not today's date; shipping_days measures order-to-shipment time.
City assignments are simulated, based on the reference snapshot rather than historical delivery evidence. Every country gets one city; source prices remain in unspecified monetary units. Original gross revenue, cost and profit remain gross measures; net revenue incorporates our fictional discounts.

In [12]:
# Read reference DATA 
lines = (META / 'geonames_countryInfo.txt').read_text().splitlines()
geo_header = next(line[1:] for line in lines if line.startswith('#ISO\t')).split('\t')
geo = pd.DataFrame(list(csv.DictReader(
    (line for line in lines if line and not line.startswith('#')),
    fieldnames=geo_header, delimiter='\t')))
aliases = json.loads((META / 'country_aliases.json').read_text())
iso_to_name = geo.set_index('ISO')['Country'].to_dict()
name_aliases = {name: iso_to_name[iso] for name, iso in aliases.items()}
cleaned['reference_country'] = cleaned['country'].replace(name_aliases)
cleaned = cleaned.merge(
    geo[['Country', 'ISO', 'Capital']].rename(columns={
        'Country': 'reference_country', 'ISO': 'country_code', 'Capital': 'shipping_city'}),
    on='reference_country', how='left', validate='many_to_one')
assert len(cleaned) == 500 and cleaned[['country_code', 'shipping_city']].notna().all().all()
cleaned['shipping_city'] = cleaned['shipping_city'].str.strip()
assert cleaned['shipping_city'].ne('').all()
cleaned = cleaned.drop(columns='reference_country')

rng = random.Random(42)
customers, coupons = [], []
for country_code in cleaned['country_code']:
    customers.append(f'C-{country_code}-{rng.randint(1, 3):03d}')
    coupons.append(rng.choice(['NONE', 'SAVE20', 'SAVE30']))
cleaned['customer_id'] = customers
cleaned['coupon_code'] = coupons
COUPONS = {'NONE': 0.0, 'SAVE20': 0.20, 'SAVE30': 0.30}
cleaned['discount_rate'] = cleaned['coupon_code'].map(COUPONS)
cleaned['net_revenue'] = [money(Decimal(str(g)) * (Decimal('1') - Decimal(str(d))))
                          for g, d in zip(cleaned['gross_revenue'], cleaned['discount_rate'])]
reference_date = cleaned['date'].max() + pd.Timedelta(days=1)
cleaned['days_since_purchase'] = (reference_date - cleaned['date']).dt.days
cleaned['shipping_days'] = (cleaned['ship_date'] - cleaned['date']).dt.days
required = {'date', 'customer_id', 'product', 'price', 'quantity', 'coupon_code', 'shipping_city'}
assert required <= set(cleaned.columns)
assert cleaned['net_revenue'].le(cleaned['gross_revenue']).all()
assert abs(cleaned['gross_revenue'].sum() - batch.total()) < 0.005
print('Unique shipping cities:', len(set(cleaned['shipping_city'])))
print('Recency reference date:', reference_date.date())
display(cleaned[['customer_id', 'coupon_code', 'shipping_city', 'days_since_purchase', 'net_revenue']].head())

Unique shipping cities: 171
Recency reference date: 2017-07-27


,customer_id,coupon_code,shipping_city,days_since_purchase,net_revenue
0,C-LY-003,NONE,Tripoli,1013,3692591.20
1,C-CA-001,SAVE30,Ottawa,2089,325467.16
2,C-LY-002,NONE,Tripoli,269,387259.76
3,C-JP-001,NONE,Tokyo,2665,683335.40
4,C-TD-003,NONE,N'Djamena,2172,91853.85


## Mini-Aggregation

Aggregate net revenue by shipping city and country. Including country prevents similarly named cities in different countries being combined. Interpret the ranking only as a result of this synthetic batch.

In [13]:
summary = (cleaned.groupby(['country', 'shipping_city'], as_index=False)
           .agg(orders=('order_id', 'size'), net_revenue=('net_revenue', 'sum'))
           .sort_values(['net_revenue', 'country'], ascending=[False, True]).reset_index(drop=True))
summary['net_revenue'] = summary['net_revenue'].round(2)
assert abs(summary['net_revenue'].sum() - cleaned['net_revenue'].sum()) < 0.005
display(summary.head(10))
top = summary.iloc[0]
share = 100 * top['net_revenue'] / cleaned['net_revenue'].sum()
insight = (f"**Insight:** {top['shipping_city']} ({top['country']}) leads this simulated batch with "
           f"{top['net_revenue']:,.2f} monetary units of net revenue across {int(top['orders'])} orders "
           f"({share:.1f}% of the total). Assigning each country's orders to its capital makes "
           "this a country ranking too; it does not establish real city-level demand.")
display(Markdown(insight))

,country,shipping_city,orders,net_revenue
0,Papua New Guinea,Port Moresby,3,14857539.78
1,Portugal,Lisbon,3,12362766.00
2,Costa Rica,San Jose,3,12016462.38
3,South Africa,Pretoria,4,11636730.03
4,Czech Republic,Prague,7,11431262.67
5,Georgia,Tbilisi,3,11205403.38
6,Swaziland,Mbabane,5,11055206.43
7,Luxembourg,Luxembourg,6,10605574.43
8,Tonga,Nuku'alofa,5,10469142.85
9,Cuba,Havana,7,10251245.11


**Insight:** Port Moresby (Papua New Guinea) leads this simulated batch with 14,857,539.78 monetary units of net revenue across 3 orders (2.5% of the total). Assigning each country's orders to its capital makes this a country ranking too; it does not establish real city-level demand.

## Serialization Checkpoint

Save both CSV and JSON, with date strings format for portability.Both contain the same 500 cleaned, enriched transactions. Reload both formats and restore types explicitly, then compare all rows and columns to the cleaned table.Verify that both files preserve the data when reloaded.

In [14]:
export = cleaned.copy()
for field in ['date', 'ship_date']:
    export[field] = export[field].dt.strftime('%Y-%m-%d')
export.to_csv(OUT / 'cleaned_sales.csv', index=False)
(OUT / 'cleaned_sales.json').write_text(
    json.dumps(export.to_dict(orient='records'), indent=2, ensure_ascii=False, allow_nan=False) + '\n')
summary.to_csv(OUT / 'revenue_by_city.csv', index=False)

def restore_types(frame):
    """Restore the analysis schema after reading either serialization format."""
    frame = frame[cleaned.columns].copy()
    for field in cleaned.columns:
        if field in ['date', 'ship_date']:
            frame[field] = pd.to_datetime(frame[field], format='%Y-%m-%d', errors='raise')
        else:
            frame[field] = frame[field].astype(cleaned[field].dtype)
    return frame

csv_back = restore_types(pd.read_csv(OUT / 'cleaned_sales.csv', dtype=str, keep_default_na=False))
json_back = restore_types(pd.DataFrame(json.loads((OUT / 'cleaned_sales.json').read_text())))
for restored in [csv_back, json_back]:
    pd.testing.assert_frame_equal(cleaned, restored, check_exact=False, rtol=0, atol=1e-9)
print('CSV and JSON round trips passed for all 500 rows.')

CSV and JSON round trips passed for all 500 rows.


## Soft Interview Reflection

Using functions helped me keep different tasks separate and run the same checks in a consistent way. make_record() created dictionaries for each transaction. issue_counts() counted the same problems before and after cleaning. The SalesBatch class kept records together and included clean() and total() methods. Cleaning the original columns before adding new synthetic features made the steps easier to follow. money() handled rounding in one place, and restore_types() helped check both exports. I learned to explain my assumptions, keep the original data safe, and tell the difference between real whitespace problems and the missing prices and duplicates that were added on purpose. Keeping metadata enrichment separate from the final data dictionary also made it clear that adding values is different from documenting what those values mean.

## Data Dictionary Section

| Field | Type | Description | Source |
| --- | --- | --- | --- |
| primary.Country | string | Sales country; original values may contain whitespace. | Supplied sales CSV header; description written for this project |
| primary.Item Type | string | Product category, not an individual SKU. | Supplied sales CSV header; description written for this project |
| primary.Order Date | date | Order date supplied in month/day/year format. | Supplied sales CSV header; description written for this project |
| primary.Order ID | string | Transaction identifier, stored as text. | Supplied sales CSV header; description written for this project |
| primary.Order Priority | string | Original priority code, retained without recoding. | Supplied sales CSV header; description written for this project |
| primary.Region | string | Geographic sales region. | Supplied sales CSV header; description written for this project |
| primary.Sales Channel | string | Online or Offline sales channel. | Supplied sales CSV header; description written for this project |
| primary.Ship Date | date | Shipment date, not delivery date. | Supplied sales CSV header; description written for this project |
| primary.Total Cost | decimal | Units Sold × Unit Cost. | Supplied sales CSV header; description written for this project |
| primary.Total Profit | decimal | Total Revenue minus Total Cost before synthetic coupons. | Supplied sales CSV header; description written for this project |
| primary.Total Revenue | decimal | Units Sold × Unit Price before synthetic coupons. | Supplied sales CSV header; description written for this project |
| primary.Unit Cost | decimal | Unit cost; monetary currency unspecified. | Supplied sales CSV header; description written for this project |
| primary.Unit Price | decimal | Unit selling price; monetary currency unspecified. | Supplied sales CSV header; description written for this project |
| primary.Units Sold | integer | Whole units sold in the order. | Supplied sales CSV header; description written for this project |
| secondary.Capital | string | Capital name listed in the reference snapshot. | [GeoNames countryInfo.txt](https://download.geonames.org/export/dump/countryInfo.txt), field Capital; definition paraphrased from source headers/documentation |
| secondary.Country | string | Country name in the reference snapshot. | [GeoNames countryInfo.txt](https://download.geonames.org/export/dump/countryInfo.txt), field Country; definition paraphrased from source headers/documentation |
| secondary.ISO | string | Two-letter country code. | [GeoNames countryInfo.txt](https://download.geonames.org/export/dump/countryInfo.txt), field ISO; definition paraphrased from source headers/documentation |
| region | string | Geographic sales region. | Supplied sales CSV: Region; project-written definition; cleaned/typed and renamed in Steps 7–8 |
| country | string | Sales country; original values may contain whitespace. | Supplied sales CSV: Country; project-written definition; cleaned/typed and renamed in Steps 7–8 |
| product | string | Product category, not an individual SKU. | Supplied sales CSV: Item Type; project-written definition; cleaned/typed and renamed in Steps 7–8 |
| sales_channel | string | Online or Offline sales channel. | Supplied sales CSV: Sales Channel; project-written definition; cleaned/typed and renamed in Steps 7–8 |
| order_priority | string | Original priority code, retained without recoding. | Supplied sales CSV: Order Priority; project-written definition; cleaned/typed and renamed in Steps 7–8 |
| date | date | Order date supplied in month/day/year format. | Supplied sales CSV: Order Date; project-written definition; cleaned/typed and renamed in Steps 7–8 |
| order_id | string | Transaction identifier, stored as text. | Supplied sales CSV: Order ID; project-written definition; cleaned/typed and renamed in Steps 7–8 |
| ship_date | date | Shipment date, not delivery date. | Supplied sales CSV: Ship Date; project-written definition; cleaned/typed and renamed in Steps 7–8 |
| quantity | integer | Whole units sold in the order. | Supplied sales CSV: Units Sold; project-written definition; cleaned/typed and renamed in Steps 7–8 |
| price | decimal | Unit selling price; monetary currency unspecified. | Supplied sales CSV: Unit Price; project-written definition; cleaned/typed and renamed in Steps 7–8 |
| unit_cost | decimal | Unit cost; monetary currency unspecified. | Supplied sales CSV: Unit Cost; project-written definition; cleaned/typed and renamed in Steps 7–8 |
| gross_revenue | decimal | Units Sold × Unit Price before synthetic coupons. | Supplied sales CSV: Total Revenue; project-written definition; cleaned/typed and renamed in Steps 7–8 |
| total_cost | decimal | Units Sold × Unit Cost. | Supplied sales CSV: Total Cost; project-written definition; cleaned/typed and renamed in Steps 7–8 |
| gross_profit | decimal | Total Revenue minus Total Cost before synthetic coupons. | Supplied sales CSV: Total Profit; project-written definition; cleaned/typed and renamed in Steps 7–8 |
| is_online | integer | Online = 1; Offline = 0. | Step 8; encode Sales Channel |
| country_code | string | Matched country code. | [GeoNames countryInfo.txt](https://download.geonames.org/export/dump/countryInfo.txt), field ISO; matched by Country and explicit aliases in Step 9 |
| shipping_city | string | Simulated destination assigned to the country capital. | [GeoNames countryInfo.txt](https://download.geonames.org/export/dump/countryInfo.txt), field Capital; synthetic country-to-capital assignment in Step 9 |
| customer_id | string | Fictional country-prefixed customer ID; repetitions allowed. | Step 9; synthetic, seed 42 |
| coupon_code | string | Fictional NONE, SAVE10 or SAVE20 code. | Step 9; synthetic, seed 42 |
| discount_rate | decimal | Coupon discount fraction: 0, 0.10 or 0.20. | Step 9; synthetic coupon mapping |
| net_revenue | decimal | gross_revenue × (1 − discount_rate), rounded to cents. | Step 9; calculated from gross revenue and synthetic discount |
| days_since_purchase | integer | Days since order relative to 2017-07-27. | Step 9; max order date + 1 day minus date |
| shipping_days | integer | Calendar days from order to shipment. | Step 9; ship_date minus date |